In [0]:
import os
from pathlib import Path
import joblib
from sklearn.cluster import DBSCAN
from sklearn.model_selection import train_test_split

import pandas as pd
pd.set_option('display.max_columns', None)
import plotly.express as px
from plotly.subplots import make_subplots

# Parametros

In [0]:
ambiente = 'project'

PORCENTAJE_SAMPLE_DATA = 1
EVERY_N_YEARS = 5
RANDOM_SEED = 0
TEST_SIZE = 0.2
N_CLUSTERS = 6

SCALER_FEATURES = [
    'wind_speed_ms', 'wind_cos_direction', 'wind_sin_direction', 'wave_height_m', 
    'wave_cos_direction', 'wave_sin_direction', 'wave_period_s', 'wave_energy', 'wave_power_kW_m'
]

EXTREME_THRESHOLD = 0.9

if 'DATABRICKS_RUNTIME_VERSION' in os.environ:
    scaler_path = f'/Volumes/cor_{ambiente}/ml/models/scaler/scaler_{{}}.pkl'
    model_path = f'/Volumes/cor_{ambiente}/ml/models/wave_clasificator/wave_clasificator_{{}}.pkl'
else:
    base_path = Path.cwd().parent
    scaler_path = f'{base_path}/scaler/scaler_{{}}.pkl'
    model_path = f'{base_path}/wave_clasificator/wave_clasificator_{{}}.pkl'

# Obtener datos

In [0]:
coast_names = (
    spark.sql(
        f"""
            SELECT DISTINCT coast_name
            FROM cor_{ambiente}.silver.swell_metrics
        """
    )
).toPandas()['coast_name'].tolist()

# DBSCAN

In [0]:
def prueba_dbscan(coast, features, extreme_features, epsilon, min_samples):
    scaled_features = [f'{f}_scaled' for f in features]
    scaled_extreme_features = [f'{f}_scaled' for f in extreme_features]
    data = (
        spark.sql(
            f"""
                SELECT coast_name, datetime, {', '.join(SCALER_FEATURES)},
                CONCAT(coast_name, '_', DATE_FORMAT(datetime, 'yyyyMM')) AS coast_year_month
                FROM cor_{ambiente}.silver.swell_metrics
                WHERE coast_name = '{coast}'
                AND YEAR(datetime) % {EVERY_N_YEARS} = 0
            """
        )
    )

    # Generar data sample
    coast_year_month_dict = {row.coast_year_month: PORCENTAJE_SAMPLE_DATA for row in data.select('coast_year_month').distinct().collect()}
    data_sample = (
        data
        .sampleBy('coast_year_month', fractions=coast_year_month_dict, seed=RANDOM_SEED)
        .drop('coast_year_month')
    ).toPandas()

    # Preparar datos
    scaler = joblib.load(scaler_path.format(coast))
    scaled_data = scaler.transform(data_sample[SCALER_FEATURES])
    scaled_df = pd.DataFrame(scaled_data, columns=SCALER_FEATURES)
    scaled_df = scaled_df[features].rename(columns=dict(zip(features, scaled_features)))

    X = pd.concat(
        [
            data_sample[features].copy(),
            scaled_df.copy()
        ],
        axis=1
    )
    
    # Identificar outliers con DBSCAN
    dbscan = DBSCAN(eps=epsilon, min_samples=min_samples)
    
    mask_outliers = dbscan.fit_predict(X[scaled_features]) == -1
    for e_f in scaled_extreme_features:
        mask_outliers &= X[e_f] > X[e_f].quantile(EXTREME_THRESHOLD)

    X['dbscan_is_outlier'] = mask_outliers

    pct = X['dbscan_is_outlier'].value_counts(normalize=True)*100
    print(pct)
    if (pct.get(True, 0) < 5) or (pct.get(True, 0) > 15):
        print(f'El modelo DBSCAN para la costa {coast} detecta un porcentaje de outliers fuera de rango ({pct.get(True, 0):.2f}%), saltando...')
        return False, X
    print(f'El modelo DBSCAN para la costa {coast} detecta un porcentaje de outliers aceptable ({pct.get(True, 0):.2f}%), continuando...')
    return True, X

In [0]:
def graficar(X, color):
    fig = px.scatter_3d(
        X,
        x='wind_speed_ms',
        y='wave_period_s',
        z='wave_height_m',
        color=color,
        opacity=0.5
    )
    fig.update_traces(marker_size=3)
    fig.update_layout(
        scene=dict(
            aspectmode='cube'
        )
    )

    return fig

In [0]:
coast = coast_names[0]
features = ['wind_speed_ms', 'wind_cos_direction', 'wind_sin_direction',
            'wave_height_m', 'wave_cos_direction', 'wave_sin_direction', 'wave_period_s']
extreme_features = ['wave_power_kW_m']
epsilon = 1
min_samples = 300

paso, X = prueba_dbscan(coast, features, extreme_features, epsilon, min_samples)
print(paso)

In [0]:
graficar(X, 'dbscan_is_outlier').show()

# Gaussian Mixture

In [0]:
def prueba_gaussianFixture(coast, features, extreme_features, epsilon, min_samples):
    scaled_features = [f'{f}_scaled' for f in features]
    scaled_extreme_features = [f'{f}_scaled' for f in extreme_features]
    data = (
        spark.sql(
            f"""
                SELECT coast_name, datetime, {', '.join(SCALER_FEATURES)},
                CONCAT(coast_name, '_', DATE_FORMAT(datetime, 'yyyyMM')) AS coast_year_month
                FROM cor_{ambiente}.silver.swell_metrics
                WHERE coast_name = '{coast}'
                AND YEAR(datetime) % {EVERY_N_YEARS} = 0
            """
        )
    )

    # Generar data sample
    coast_year_month_dict = {row.coast_year_month: PORCENTAJE_SAMPLE_DATA for row in data.select('coast_year_month').distinct().collect()}
    data_sample = (
        data
        .sampleBy('coast_year_month', fractions=coast_year_month_dict, seed=RANDOM_SEED)
        .drop('coast_year_month')
    ).toPandas()

    # Preparar datos
    scaler = joblib.load(scaler_path.format(coast))
    scaled_data = scaler.transform(data_sample[SCALER_FEATURES])
    scaled_df = pd.DataFrame(scaled_data, columns=SCALER_FEATURES)
    scaled_df = scaled_df[features].rename(columns=dict(zip(features, scaled_features)))

    X = pd.concat(
        [
            data_sample[features].copy(),
            scaled_df.copy()
        ],
        axis=1
    )
    
    # Identificar outliers con DBSCAN
    dbscan = DBSCAN(eps=epsilon, min_samples=min_samples)
    
    mask_outliers = dbscan.fit_predict(X[scaled_features]) == -1
    #for e_f in scaled_extreme_features:
    #    mask_outliers &= X[e_f] > X[e_f].quantile(EXTREME_THRESHOLD)

    X['dbscan_is_outlier'] = mask_outliers

    pct = X['dbscan_is_outlier'].value_counts(normalize=True)*100
    if pct.get(True, 0) > 10:
        print(f'El modelo DBSCAN para la costa {coast} detecta un porcentaje de outliers fuera de rango ({pct.get(True, 0):.2f}%), saltando...')
        return False
    print(f'El modelo DBSCAN para la costa {coast} detecta un porcentaje de outliers aceptable ({pct.get(True, 0):.2f}%), continuando...')
    
    X_train, X_test = train_test_split(X, test_size=TEST_SIZE, random_state=RANDOM_SEED)